# 01 Data Loading and Combining

Notebook ini digunakan untuk membaca seluruh file dataset penjualan e-commerce Indonesia periode Januari 2024 sampai November 2025.

Tahapan utama pada notebook ini meliputi:

1. Membaca semua file Excel dari folder `datasets`.
2. Mengecek jumlah file dan struktur kolom.
3. Mengambil informasi bulan dan tahun dari nama file.
4. Menggabungkan seluruh file menjadi satu dataset utama.
5. Melakukan pengecekan awal seperti jumlah baris, kolom, missing value, dan duplikasi.
6. Menyimpan dataset gabungan ke format CSV.
7. Membaca ulang dataset gabungan menggunakan PySpark.
8. Menyimpan dataset ke format Parquet untuk kebutuhan pemrosesan Big Data.

## Import Library

In [2]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Inisialisasi Direktori Project

In [3]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATASET_DIR = BASE_DIR / "datasets"
COMBINED_DIR = DATASET_DIR / "combined"

COMBINED_DIR.mkdir(parents=True, exist_ok=True)

print("Current directory :", CURRENT_DIR)
print("Base directory    :", BASE_DIR)
print("Dataset directory :", DATASET_DIR)
print("Combined directory:", COMBINED_DIR)

Current directory : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\notebooks
Base directory    : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code
Dataset directory : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\datasets
Combined directory: e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\datasets\combined


## Mengecek File Excel Dataset

In [4]:
excel_files = list(DATASET_DIR.glob("*Sales*_clean.xlsx"))

print("Jumlah file Excel ditemukan:", len(excel_files))

for file in excel_files:
    print("-", file.name)

Jumlah file Excel ditemukan: 23
- AprilSales2024_clean.xlsx
- AprilSales2025_clean.xlsx
- AugustSales2024_clean.xlsx
- AugustSales2025_clean.xlsx
- DecemberSales2024_clean.xlsx
- FebruarySales2024_clean.xlsx
- FebruarySales2025_clean.xlsx
- JanuarySales2024_clean.xlsx
- JanuarySales2025_clean.xlsx
- JulySales2024_clean.xlsx
- JulySales2025_clean.xlsx
- JuneSales2024_clean.xlsx
- JuneSales2025_clean.xlsx
- MarchSales2024_clean.xlsx
- MarchSales2025_clean.xlsx
- MaySales2024_clean.xlsx
- MaySales2025_clean.xlsx
- NovemberSales2024_clean.xlsx
- NovemberSales2025_clean.xlsx
- OctoberSales2024_clean.xlsx
- OctoberSales2025_clean.xlsx
- SeptemberSales2024_clean.xlsx
- SeptemberSales2025_clean.xlsx


## Mapping Bulan

In [5]:
month_map = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}

month_name_id = {
    1: "Januari",
    2: "Februari",
    3: "Maret",
    4: "April",
    5: "Mei",
    6: "Juni",
    7: "Juli",
    8: "Agustus",
    9: "September",
    10: "Oktober",
    11: "November",
    12: "Desember"
}

## Fungsi Mengambil Bulan dan Tahun dari Nama File

In [6]:
def extract_period_from_filename(file_name):
    """
    Fungsi untuk mengambil informasi bulan dan tahun dari nama file.

    Contoh:
    JanuarySales2024_clean.xlsx -> bulan = 1, tahun = 2024
    NovemberSales2025_clean.xlsx -> bulan = 11, tahun = 2025
    """
    
    file_name = Path(file_name).name
    pattern = r"([A-Za-z]+)Sales(\d{4})_clean\.xlsx"
    
    match = re.match(pattern, file_name)
    
    if match is None:
        return None, None, None
    
    month_english = match.group(1)
    year = int(match.group(2))
    month_number = month_map.get(month_english)
    
    return month_number, month_english, year

## Mengurutkan File Berdasarkan Tahun dan Bulan

In [7]:
file_periods = []

for file in excel_files:
    month_number, month_english, year = extract_period_from_filename(file.name)
    
    file_periods.append({
        "file_path": file,
        "file_name": file.name,
        "year": year,
        "month": month_number,
        "month_english": month_english,
        "month_indonesia": month_name_id.get(month_number),
        "period": f"{year}-{month_number:02d}" if year is not None and month_number is not None else None
    })

file_periods_df = pd.DataFrame(file_periods)
file_periods_df = file_periods_df.sort_values(["year", "month"]).reset_index(drop=True)

excel_files = file_periods_df["file_path"].tolist()

file_periods_df[["file_name", "year", "month", "month_indonesia", "period"]]

,file_name,year,month,month_indonesia,period
0,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01
1,FebruarySales2024_clean.xlsx,2024,2,Februari,2024-02
2,MarchSales2024_clean.xlsx,2024,3,Maret,2024-03
3,AprilSales2024_clean.xlsx,2024,4,April,2024-04
4,MaySales2024_clean.xlsx,2024,5,Mei,2024-05
5,JuneSales2024_clean.xlsx,2024,6,Juni,2024-06
6,JulySales2024_clean.xlsx,2024,7,Juli,2024-07
7,AugustSales2024_clean.xlsx,2024,8,Agustus,2024-08
8,SeptemberSales2024_clean.xlsx,2024,9,September,2024-09
9,OctoberSales2024_clean.xlsx,2024,10,Oktober,2024-10


## Validasi Jumlah File

In [8]:
expected_total_files = 23

print("Jumlah file ditemukan:", len(excel_files))
print("Jumlah file yang diharapkan:", expected_total_files)

if len(excel_files) == expected_total_files:
    print("Status: Jumlah file sudah sesuai.")
else:
    print("Status: Jumlah file belum sesuai. Silakan cek kembali folder datasets.")

Jumlah file ditemukan: 23
Jumlah file yang diharapkan: 23
Status: Jumlah file sudah sesuai.


## Mengecek Struktur Kolom Setiap File

In [9]:
column_check = []

for file in excel_files:
    sample_df = pd.read_excel(file, nrows=5)
    
    month_number, month_english, year = extract_period_from_filename(file.name)
    
    column_check.append({
        "file_name": file.name,
        "year": year,
        "month": month_number,
        "total_columns": len(sample_df.columns),
        "columns": list(sample_df.columns)
    })

column_check_df = pd.DataFrame(column_check)

column_check_df[["file_name", "year", "month", "total_columns"]]

,file_name,year,month,total_columns
0,JanuarySales2024_clean.xlsx,2024,1,18
1,FebruarySales2024_clean.xlsx,2024,2,18
2,MarchSales2024_clean.xlsx,2024,3,18
3,AprilSales2024_clean.xlsx,2024,4,18
4,MaySales2024_clean.xlsx,2024,5,18
5,JuneSales2024_clean.xlsx,2024,6,18
6,JulySales2024_clean.xlsx,2024,7,18
7,AugustSales2024_clean.xlsx,2024,8,18
8,SeptemberSales2024_clean.xlsx,2024,9,18
9,OctoberSales2024_clean.xlsx,2024,10,18


## Menampilkan Seluruh Kolom Unik

In [10]:
all_columns = []

for item in column_check:
    all_columns.extend(item["columns"])

unique_columns = list(dict.fromkeys(all_columns))

print("Jumlah kolom unik:", len(unique_columns))
print("\nDaftar kolom unik:")

for col in unique_columns:
    print("-", col)

Jumlah kolom unik: 18

Daftar kolom unik:
- order_id
- total_qty
- total_weight_gr
- total_returned_qty
- Total Diskon
- product_categories
- num_product_categories
- Status Pesanan
- Alasan Pembatalan
- Opsi Pengiriman
- Metode Pembayaran
- Kota/Kabupaten
- Provinsi
- Ongkos Kirim Dibayar oleh Pembeli
- Estimasi Potongan Biaya Pengiriman
- Total Pembayaran
- Perkiraan Ongkos Kirim
- Waktu Pesanan Dibuat


## Mengecek Perbedaan Kolom Antar File

In [11]:
reference_columns = unique_columns
column_difference = []

for item in column_check:
    current_columns = set(item["columns"])
    reference_set = set(reference_columns)
    
    missing_columns = sorted(list(reference_set - current_columns))
    extra_columns = sorted(list(current_columns - reference_set))
    
    column_difference.append({
        "file_name": item["file_name"],
        "total_columns": item["total_columns"],
        "missing_columns": missing_columns,
        "total_missing_columns": len(missing_columns),
        "extra_columns": extra_columns,
        "total_extra_columns": len(extra_columns)
    })

column_difference_df = pd.DataFrame(column_difference)

column_difference_df[
    [
        "file_name",
        "total_columns",
        "total_missing_columns",
        "missing_columns"
    ]
]

,file_name,total_columns,total_missing_columns,missing_columns
0,JanuarySales2024_clean.xlsx,18,0,[]
1,FebruarySales2024_clean.xlsx,18,0,[]
2,MarchSales2024_clean.xlsx,18,0,[]
3,AprilSales2024_clean.xlsx,18,0,[]
4,MaySales2024_clean.xlsx,18,0,[]
5,JuneSales2024_clean.xlsx,18,0,[]
6,JulySales2024_clean.xlsx,18,0,[]
7,AugustSales2024_clean.xlsx,18,0,[]
8,SeptemberSales2024_clean.xlsx,18,0,[]
9,OctoberSales2024_clean.xlsx,18,0,[]


## Fungsi Pembersihan Nama Kolom

In [12]:
def clean_column_names(df):
    """
    Fungsi untuk membersihkan nama kolom agar lebih konsisten.
    Pada tahap ini, nama kolom tidak diubah menjadi lowercase karena beberapa kolom
    masih ingin dipertahankan sesuai bentuk aslinya untuk memudahkan pengecekan.
    """
    
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]
    
    return df

## Membaca dan Menggabungkan Semua File Excel

In [13]:
combined_list = []

for file in excel_files:
    print("Membaca file:", file.name)
    
    df = pd.read_excel(file)
    df = clean_column_names(df)
    
    month_number, month_english, year = extract_period_from_filename(file.name)
    
    df["source_file"] = file.name
    df["source_year"] = year
    df["source_month"] = month_number
    df["source_month_name"] = month_name_id.get(month_number)
    df["source_period"] = f"{year}-{month_number:02d}"
    
    combined_list.append(df)

df_combined = pd.concat(combined_list, ignore_index=True)

print("\nDataset berhasil digabung.")
print("Jumlah baris:", df_combined.shape[0])
print("Jumlah kolom:", df_combined.shape[1])

df_combined.head()

Membaca file: JanuarySales2024_clean.xlsx
Membaca file: FebruarySales2024_clean.xlsx
Membaca file: MarchSales2024_clean.xlsx
Membaca file: AprilSales2024_clean.xlsx
Membaca file: MaySales2024_clean.xlsx
Membaca file: JuneSales2024_clean.xlsx
Membaca file: JulySales2024_clean.xlsx
Membaca file: AugustSales2024_clean.xlsx
Membaca file: SeptemberSales2024_clean.xlsx
Membaca file: OctoberSales2024_clean.xlsx
Membaca file: NovemberSales2024_clean.xlsx
Membaca file: DecemberSales2024_clean.xlsx
Membaca file: JanuarySales2025_clean.xlsx
Membaca file: FebruarySales2025_clean.xlsx
Membaca file: MarchSales2025_clean.xlsx
Membaca file: AprilSales2025_clean.xlsx
Membaca file: MaySales2025_clean.xlsx
Membaca file: JuneSales2025_clean.xlsx
Membaca file: JulySales2025_clean.xlsx
Membaca file: AugustSales2025_clean.xlsx
Membaca file: SeptemberSales2025_clean.xlsx
Membaca file: OctoberSales2025_clean.xlsx
Membaca file: NovemberSales2025_clean.xlsx

Dataset berhasil digabung.
Jumlah baris: 20404
Jumlah 

,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,source_file,source_year,source_month,source_month_name,source_period
0,ORD_0006556,2,50,0,0,Aksesoris ID Card,1,Selesai,NaN,Hemat Kargo-SPX Hemat,Saldo ShopeePay,KAB. BEKASI,JAWA BARAT,0,8000,10000,8000,2024-01-01 00:05,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01
1,ORD_0006557,2,1000,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA BANDUNG,JAWA BARAT,0,10000,35663,10000,2024-01-01 00:07,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01
2,ORD_0006558,1,600,0,0,Plastik / Wadah Plastik,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,COD (Bayar di Tempat),KAB. BOGOR,JAWA BARAT,0,10000,23187,10000,2024-01-01 00:07,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01
3,ORD_0006559,1,700,0,0,Rak / Rak Serbaguna,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA DEPOK,JAWA BARAT,0,8000,37400,8000,2024-01-01 00:12,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01
4,ORD_0006560,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,Online Payment,KAB. BEKASI,JAWA BARAT,0,8000,21800,8000,2024-01-01 00:36,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01


## Informasi Awal Dataset Gabungan

In [14]:
print("Informasi dataset gabungan")
print("=" * 50)
print("Jumlah baris :", df_combined.shape[0])
print("Jumlah kolom :", df_combined.shape[1])
print("=" * 50)

df_combined.head()

Informasi dataset gabungan
Jumlah baris : 20404
Jumlah kolom : 23


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,source_file,source_year,source_month,source_month_name,source_period
0,ORD_0006556,2,50,0,0,Aksesoris ID Card,1,Selesai,NaN,Hemat Kargo-SPX Hemat,Saldo ShopeePay,KAB. BEKASI,JAWA BARAT,0,8000,10000,8000,2024-01-01 00:05,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01
1,ORD_0006557,2,1000,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA BANDUNG,JAWA BARAT,0,10000,35663,10000,2024-01-01 00:07,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01
2,ORD_0006558,1,600,0,0,Plastik / Wadah Plastik,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,COD (Bayar di Tempat),KAB. BOGOR,JAWA BARAT,0,10000,23187,10000,2024-01-01 00:07,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01
3,ORD_0006559,1,700,0,0,Rak / Rak Serbaguna,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA DEPOK,JAWA BARAT,0,8000,37400,8000,2024-01-01 00:12,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01
4,ORD_0006560,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,Online Payment,KAB. BEKASI,JAWA BARAT,0,8000,21800,8000,2024-01-01 00:36,JanuarySales2024_clean.xlsx,2024,1,Januari,2024-01


## Menampilkan Daftar Kolom Dataset Gabungan

In [15]:
for i, col in enumerate(df_combined.columns, start=1):
    print(f"{i}. {col}")

1. order_id
2. total_qty
3. total_weight_gr
4. total_returned_qty
5. Total Diskon
6. product_categories
7. num_product_categories
8. Status Pesanan
9. Alasan Pembatalan
10. Opsi Pengiriman
11. Metode Pembayaran
12. Kota/Kabupaten
13. Provinsi
14. Ongkos Kirim Dibayar oleh Pembeli
15. Estimasi Potongan Biaya Pengiriman
16. Total Pembayaran
17. Perkiraan Ongkos Kirim
18. Waktu Pesanan Dibuat
19. source_file
20. source_year
21. source_month
22. source_month_name
23. source_period


## Mengecek Tipe Data Awal

In [16]:
df_combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20404 entries, 0 to 20403
Data columns (total 23 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   order_id                            20404 non-null  object
 1   total_qty                           20404 non-null  int64 
 2   total_weight_gr                     20404 non-null  int64 
 3   total_returned_qty                  20404 non-null  int64 
 4   Total Diskon                        20404 non-null  int64 
 5   product_categories                  20404 non-null  object
 6   num_product_categories              20404 non-null  int64 
 7   Status Pesanan                      20404 non-null  object
 8   Alasan Pembatalan                   2738 non-null   object
 9   Opsi Pengiriman                     20404 non-null  object
 10  Metode Pembayaran                   20404 non-null  object
 11  Kota/Kabupaten                      20404 non-null  ob

## Mengecek Jumlah Data per File

In [17]:
rows_per_file = (
    df_combined
    .groupby(["source_year", "source_month", "source_month_name", "source_file"])
    .size()
    .reset_index(name="total_rows")
    .sort_values(["source_year", "source_month"])
)

rows_per_file

,source_year,source_month,source_month_name,source_file,total_rows
0,2024,1,Januari,JanuarySales2024_clean.xlsx,431
1,2024,2,Februari,FebruarySales2024_clean.xlsx,389
2,2024,3,Maret,MarchSales2024_clean.xlsx,453
3,2024,4,April,AprilSales2024_clean.xlsx,492
4,2024,5,Mei,MaySales2024_clean.xlsx,620
5,2024,6,Juni,JuneSales2024_clean.xlsx,697
6,2024,7,Juli,JulySales2024_clean.xlsx,1270
7,2024,8,Agustus,AugustSales2024_clean.xlsx,1293
8,2024,9,September,SeptemberSales2024_clean.xlsx,1123
9,2024,10,Oktober,OctoberSales2024_clean.xlsx,1217


## Mengecek Jumlah Data per Bulan

In [18]:
rows_per_month = (
    df_combined
    .groupby(["source_year", "source_month", "source_month_name", "source_period"])
    .size()
    .reset_index(name="total_transactions")
    .sort_values(["source_year", "source_month"])
)

rows_per_month

,source_year,source_month,source_month_name,source_period,total_transactions
0,2024,1,Januari,2024-01,431
1,2024,2,Februari,2024-02,389
2,2024,3,Maret,2024-03,453
3,2024,4,April,2024-04,492
4,2024,5,Mei,2024-05,620
5,2024,6,Juni,2024-06,697
6,2024,7,Juli,2024-07,1270
7,2024,8,Agustus,2024-08,1293
8,2024,9,September,2024-09,1123
9,2024,10,Oktober,2024-10,1217


## Mengecek Missing Value

In [19]:
missing_values = df_combined.isnull().sum().reset_index()
missing_values.columns = ["column", "missing_count"]
missing_values["missing_percentage"] = (
    missing_values["missing_count"] / len(df_combined) * 100
)

missing_values = missing_values.sort_values("missing_count", ascending=False)

missing_values

,column,missing_count,missing_percentage
8,Alasan Pembatalan,17666,86.581063
17,Waktu Pesanan Dibuat,1980,9.703980
0,order_id,0,0.000000
12,Provinsi,0,0.000000
21,source_month_name,0,0.000000
20,source_month,0,0.000000
19,source_year,0,0.000000
18,source_file,0,0.000000
16,Perkiraan Ongkos Kirim,0,0.000000
15,Total Pembayaran,0,0.000000


## Mengecek Duplikasi Data

In [20]:
total_duplicate_rows = df_combined.duplicated().sum()

print("Jumlah baris duplikat:", total_duplicate_rows)

Jumlah baris duplikat: 0


## Mengecek Duplikasi Berdasarkan order_id

In [21]:
if "order_id" in df_combined.columns:
    duplicate_order_id = df_combined["order_id"].duplicated().sum()
    
    print("Jumlah duplikasi berdasarkan order_id:", duplicate_order_id)
else:
    print("Kolom order_id tidak ditemukan.")

Jumlah duplikasi berdasarkan order_id: 0


## Mengecek Status Pesanan

In [22]:
if "Status Pesanan" in df_combined.columns:
    status_order = df_combined["Status Pesanan"].value_counts(dropna=False).reset_index()
    status_order.columns = ["Status Pesanan", "total"]
    display(status_order)
else:
    print("Kolom Status Pesanan tidak ditemukan.")

,Status Pesanan,total
0,Selesai,17416
1,Batal,2738
2,Sedang Dikirim,59
3,"Pesanan diterima, namun Pembeli masih dapat me...",38
4,"Pesanan diterima, namun Pembeli masih dapat me...",34
5,"Pesanan diterima, namun Pembeli masih dapat me...",34
6,Telah Dikirim,30
7,"Pesanan diterima, namun Pembeli masih dapat me...",25
8,"Pesanan diterima, namun Pembeli masih dapat me...",22
9,"Pesanan diterima, namun Pembeli masih dapat me...",6


## Mengecek Alasan Pembatalan

In [23]:
if "Alasan Pembatalan" in df_combined.columns:
    cancellation_reason = (
        df_combined["Alasan Pembatalan"]
        .value_counts(dropna=False)
        .reset_index()
    )
    
    cancellation_reason.columns = ["Alasan Pembatalan", "total"]
    display(cancellation_reason.head(20))
else:
    print("Kolom Alasan Pembatalan tidak ditemukan.")

,Alasan Pembatalan,total
0,NaN,17666
1,Dibatalkan oleh Pembeli. Alasan: Lainnya/ beru...,639
2,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,543
3,Dibatalkan secara otomatis oleh sistem. Alasan...,431
4,Dibatalkan oleh Pembeli. Alasan: Need to chang...,302
5,Dibatalkan secara otomatis oleh sistem. Alasan...,247
6,Dibatalkan secara otomatis oleh sistem. Alasan...,113
7,Dibatalkan oleh Pembeli. Alasan: Perlu menguba...,101
8,Dibatalkan oleh Pembeli. Alasan: Perlu menguba...,77
9,Dibatalkan secara otomatis oleh sistem. Alasan...,63


## Fungsi Konversi Kolom Numerik

In [24]:
def convert_to_numeric(series):
    """
    Fungsi untuk mengubah kolom menjadi numerik.
    Jika data sudah numerik, maka langsung dikonversi.
    Jika data berbentuk teks, beberapa karakter umum akan dibersihkan terlebih dahulu.
    """
    
    if series.dtype == "object":
        series = (
            series.astype(str)
            .str.replace("Rp", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace("-", "0", regex=False)
        )
    
    return pd.to_numeric(series, errors="coerce")

## Membersihkan Kolom Numerik

In [25]:
numeric_columns = [
    "total_qty",
    "total_weight_gr",
    "total_returned_qty",
    "Total Diskon",
    "num_product_categories",
    "Ongkos Kirim Dibayar oleh Pembeli",
    "Estimasi Potongan Biaya Pengiriman",
    "Total Pembayaran",
    "Perkiraan Ongkos Kirim"
]

existing_numeric_columns = [
    col for col in numeric_columns 
    if col in df_combined.columns
]

for col in existing_numeric_columns:
    df_combined[col] = convert_to_numeric(df_combined[col])

df_combined[existing_numeric_columns].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20404 entries, 0 to 20403
Data columns (total 9 columns):
 #   Column                              Non-Null Count  Dtype
---  ------                              --------------  -----
 0   total_qty                           20404 non-null  int64
 1   total_weight_gr                     20404 non-null  int64
 2   total_returned_qty                  20404 non-null  int64
 3   Total Diskon                        20404 non-null  int64
 4   num_product_categories              20404 non-null  int64
 5   Ongkos Kirim Dibayar oleh Pembeli   20404 non-null  int64
 6   Estimasi Potongan Biaya Pengiriman  20404 non-null  int64
 7   Total Pembayaran                    20404 non-null  int64
 8   Perkiraan Ongkos Kirim              20404 non-null  int64
dtypes: int64(9)
memory usage: 1.4 MB


## Ringkasan Statistik Kolom Numerik

In [26]:
df_combined[existing_numeric_columns].describe()

,total_qty,total_weight_gr,total_returned_qty,Total Diskon,num_product_categories,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim
count,20404.000000,20404.000000,20404.000000,20404.000000,20404.000000,20404.000000,20404.000000,2.040400e+04,20404.000000
mean,2.561998,2009.382719,0.018134,414.016663,1.111645,4086.332386,10840.496912,5.061760e+04,18424.159871
std,7.861694,7161.302818,0.554587,9889.717663,0.485423,13507.052748,12713.400864,1.446010e+05,23862.751497
min,1.000000,10.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000e+00,1.000000
25%,1.000000,300.000000,0.000000,0.000000,1.000000,0.000000,3500.000000,1.348000e+04,8000.000000
50%,1.000000,500.000000,0.000000,0.000000,1.000000,0.000000,9500.000000,2.159200e+04,11500.000000
75%,2.000000,1600.000000,0.000000,0.000000,1.000000,2500.000000,15000.000000,4.156000e+04,20000.000000
max,256.000000,375000.000000,70.000000,700000.000000,11.000000,584000.000000,312000.000000,3.403591e+06,959200.000000


## Membuat Kolom net_qty

In [27]:
if "total_qty" in df_combined.columns:
    if "total_returned_qty" in df_combined.columns:
        df_combined["total_returned_qty"] = df_combined["total_returned_qty"].fillna(0)
        df_combined["net_qty"] = df_combined["total_qty"] - df_combined["total_returned_qty"]
    else:
        df_combined["net_qty"] = df_combined["total_qty"]
    
    print("Kolom net_qty berhasil dibuat.")
else:
    print("Kolom total_qty tidak ditemukan, sehingga net_qty tidak dapat dibuat.")

df_combined[["total_qty", "total_returned_qty", "net_qty"]].head()

Kolom net_qty berhasil dibuat.


,total_qty,total_returned_qty,net_qty
0,2,0,2
1,2,0,2
2,1,0,1
3,1,0,1
4,1,0,1


## Mengecek Distribusi total_qty dan net_qty

In [28]:
quantity_columns = ["total_qty", "net_qty"]

existing_quantity_columns = [
    col for col in quantity_columns 
    if col in df_combined.columns
]

df_combined[existing_quantity_columns].describe()

,total_qty,net_qty
count,20404.000000,20404.000000
mean,2.561998,2.543864
std,7.861694,7.818901
min,1.000000,0.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,256.000000,256.000000


## Membuat Kolom Penanda Pesanan Batal

In [29]:
if "Status Pesanan" in df_combined.columns:
    df_combined["is_cancelled"] = np.where(
        df_combined["Status Pesanan"].astype(str).str.lower().str.contains("batal"),
        1,
        0
    )
    
    print("Kolom is_cancelled berhasil dibuat.")
    display(df_combined["is_cancelled"].value_counts().reset_index())
else:
    print("Kolom Status Pesanan tidak ditemukan.")

Kolom is_cancelled berhasil dibuat.


,is_cancelled,count
0,0,17666
1,1,2738


## Mengecek Kolom Waktu Pesanan

In [30]:
if "Waktu Pesanan Dibuat" in df_combined.columns:
    total_missing_order_time = df_combined["Waktu Pesanan Dibuat"].isnull().sum()
    
    print("Jumlah missing pada kolom Waktu Pesanan Dibuat:", total_missing_order_time)
    
    missing_time_by_file = (
        df_combined[df_combined["Waktu Pesanan Dibuat"].isnull()]
        .groupby("source_file")
        .size()
        .reset_index(name="missing_order_time")
        .sort_values("missing_order_time", ascending=False)
    )
    
    display(missing_time_by_file)
else:
    print("Kolom Waktu Pesanan Dibuat tidak ditemukan.")

Jumlah missing pada kolom Waktu Pesanan Dibuat: 1980


,source_file,missing_order_time
0,DecemberSales2024_clean.xlsx,1214
1,JulySales2025_clean.xlsx,766


## Membuat Fitur Waktu Berdasarkan Sumber File

In [31]:
# Karena beberapa file tidak memiliki kolom Waktu Pesanan Dibuat,
# fitur tahun dan bulan utama tetap dibuat dari nama file.

df_combined["order_year"] = df_combined["source_year"]
df_combined["order_month"] = df_combined["source_month"]

df_combined[["source_file", "order_year", "order_month", "source_period"]].head()

,source_file,order_year,order_month,source_period
0,JanuarySales2024_clean.xlsx,2024,1,2024-01
1,JanuarySales2024_clean.xlsx,2024,1,2024-01
2,JanuarySales2024_clean.xlsx,2024,1,2024-01
3,JanuarySales2024_clean.xlsx,2024,1,2024-01
4,JanuarySales2024_clean.xlsx,2024,1,2024-01


## Cek Kategori Produk Terbanyak

In [32]:
if "product_categories" in df_combined.columns:
    top_product_categories = (
        df_combined["product_categories"]
        .value_counts(dropna=False)
        .reset_index()
    )
    
    top_product_categories.columns = ["product_categories", "total"]
    display(top_product_categories.head(20))
else:
    print("Kolom product_categories tidak ditemukan.")

,product_categories,total
0,Celengan,5342
1,Mangkok Sambal / Saus,3520
2,Aksesoris Pintu,2721
3,Nampan / Tray,1300
4,Seal / Baut / Roof,652
5,Baskom / Mangkok Besar,573
6,Rak / Rak Serbaguna,568
7,Lunch Box / Rantang,522
8,Keranjang,493
9,Other,493


## Cek Provinsi Terbanyak

In [33]:
if "Provinsi" in df_combined.columns:
    top_provinces = (
        df_combined["Provinsi"]
        .value_counts(dropna=False)
        .reset_index()
    )
    
    top_provinces.columns = ["Provinsi", "total"]
    display(top_provinces.head(20))
else:
    print("Kolom Provinsi tidak ditemukan.")

,Provinsi,total
0,JAWA BARAT,6497
1,BANTEN,3528
2,DKI JAKARTA,2811
3,JAWA TIMUR,1587
4,JAWA TENGAH,1477
5,SUMATERA SELATAN,625
6,LAMPUNG,483
7,JAMBI,338
8,RIAU,333
9,SUMATERA UTARA,306


## Cek Metode Pembayaran Terbanyak

In [34]:
if "Metode Pembayaran" in df_combined.columns:
    top_payment_methods = (
        df_combined["Metode Pembayaran"]
        .value_counts(dropna=False)
        .reset_index()
    )
    
    top_payment_methods.columns = ["Metode Pembayaran", "total"]
    display(top_payment_methods.head(20))
else:
    print("Kolom Metode Pembayaran tidak ditemukan.")

,Metode Pembayaran,total
0,COD (Bayar di Tempat),11332
1,Saldo ShopeePay,3599
2,Online Payment,3109
3,SPayLater,1456
4,SeaBank Bayar Instan,584
5,Kartu Kredit/Debit,222
6,Indomaret/i.Saku,42
7,Alfamart/Alfamidi/Dan+Dan,35
8,Pembayaran dibebaskan,18
9,BCA OneKlik,3


## Cek Opsi Pengiriman Terbanyak

In [35]:
if "Opsi Pengiriman" in df_combined.columns:
    top_shipping_options = (
        df_combined["Opsi Pengiriman"]
        .value_counts(dropna=False)
        .reset_index()
    )
    
    top_shipping_options.columns = ["Opsi Pengiriman", "total"]
    display(top_shipping_options.head(20))
else:
    print("Kolom Opsi Pengiriman tidak ditemukan.")

,Opsi Pengiriman,total
0,Hemat Kargo-SPX Hemat,12306
1,Reguler (Cashless)-SPX Standard,4587
2,SPX Hemat,543
3,Hemat Kargo,433
4,Reguler (Cashless)-JNE Reguler,340
5,Instant (Versi Lama)-SPX Instant (Versi Lama),327
6,Same Day-SPX Sameday,302
7,SPX Standard,192
8,Reguler (Cashless),184
9,Kargo-JNE Trucking (JTR),173


## Menyimpan Dataset Gabungan ke CSV

In [36]:
combined_csv_path = COMBINED_DIR / "indonesia_ecommerce_sales_2024_2025_combined.csv"

df_combined.to_csv(combined_csv_path, index=False, encoding="utf-8-sig")

print("Dataset gabungan berhasil disimpan ke:")
print(combined_csv_path)

Dataset gabungan berhasil disimpan ke:
e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\datasets\combined\indonesia_ecommerce_sales_2024_2025_combined.csv


## Import Pyspark

In [37]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    count,
    when,
    isnan,
    sum as spark_sum
)

## Membuat Spark Session

In [38]:
spark = (
    SparkSession.builder
    .appName("Indonesia E-Commerce Sales 2024-2025")
    .getOrCreate()
)

spark

## Membaca Dataset Gabungan dengan PySpark

In [39]:
spark_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(str(combined_csv_path))
)

print("Jumlah baris Spark DataFrame:", spark_df.count())
print("Jumlah kolom Spark DataFrame:", len(spark_df.columns))

Jumlah baris Spark DataFrame: 20404
Jumlah kolom Spark DataFrame: 27


## Menampilkan Schema Dataset Spark

In [40]:
spark_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- total_qty: integer (nullable = true)
 |-- total_weight_gr: integer (nullable = true)
 |-- total_returned_qty: integer (nullable = true)
 |-- Total Diskon: integer (nullable = true)
 |-- product_categories: string (nullable = true)
 |-- num_product_categories: integer (nullable = true)
 |-- Status Pesanan: string (nullable = true)
 |-- Alasan Pembatalan: string (nullable = true)
 |-- Opsi Pengiriman: string (nullable = true)
 |-- Metode Pembayaran: string (nullable = true)
 |-- Kota/Kabupaten: string (nullable = true)
 |-- Provinsi: string (nullable = true)
 |-- Ongkos Kirim Dibayar oleh Pembeli: integer (nullable = true)
 |-- Estimasi Potongan Biaya Pengiriman: integer (nullable = true)
 |-- Total Pembayaran: integer (nullable = true)
 |-- Perkiraan Ongkos Kirim: integer (nullable = true)
 |-- Waktu Pesanan Dibuat: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_year: integer (nullable = true)
 |-- sou

## Menampilkan Data Awal dengan Spark

In [41]:
spark_df.show(5, truncate=False)

+-----------+---------+---------------+------------------+------------+-----------------------+----------------------+--------------+-----------------+-------------------------------+---------------------+--------------+----------+---------------------------------+----------------------------------+----------------+----------------------+--------------------+---------------------------+-----------+------------+-----------------+-------------------+-------+------------+----------+-----------+
|order_id   |total_qty|total_weight_gr|total_returned_qty|Total Diskon|product_categories     |num_product_categories|Status Pesanan|Alasan Pembatalan|Opsi Pengiriman                |Metode Pembayaran    |Kota/Kabupaten|Provinsi  |Ongkos Kirim Dibayar oleh Pembeli|Estimasi Potongan Biaya Pengiriman|Total Pembayaran|Perkiraan Ongkos Kirim|Waktu Pesanan Dibuat|source_file                |source_year|source_month|source_month_name|source_period      |net_qty|is_cancelled|order_year|order_month|
+-----

## Mengecek Missing Value dengan Spark

In [42]:
missing_expr = []

for column_name in spark_df.columns:
    missing_expr.append(
        count(
            when(
                col(column_name).isNull(),
                column_name
            )
        ).alias(column_name)
    )

spark_missing_df = spark_df.select(missing_expr)

spark_missing_df.show(truncate=False)

+--------+---------+---------------+------------------+------------+------------------+----------------------+--------------+-----------------+---------------+-----------------+--------------+--------+---------------------------------+----------------------------------+----------------+----------------------+--------------------+-----------+-----------+------------+-----------------+-------------+-------+------------+----------+-----------+
|order_id|total_qty|total_weight_gr|total_returned_qty|Total Diskon|product_categories|num_product_categories|Status Pesanan|Alasan Pembatalan|Opsi Pengiriman|Metode Pembayaran|Kota/Kabupaten|Provinsi|Ongkos Kirim Dibayar oleh Pembeli|Estimasi Potongan Biaya Pengiriman|Total Pembayaran|Perkiraan Ongkos Kirim|Waktu Pesanan Dibuat|source_file|source_year|source_month|source_month_name|source_period|net_qty|is_cancelled|order_year|order_month|
+--------+---------+---------------+------------------+------------+------------------+----------------------+

## Jumlah Transaksi per Bulan dengan Spark

In [43]:
(
    spark_df
    .groupBy("source_year", "source_month", "source_month_name", "source_period")
    .count()
    .orderBy("source_year", "source_month")
    .show(30, truncate=False)
)

+-----------+------------+-----------------+-------------------+-----+
|source_year|source_month|source_month_name|source_period      |count|
+-----------+------------+-----------------+-------------------+-----+
|2024       |1           |Januari          |2024-01-01 00:00:00|431  |
|2024       |2           |Februari         |2024-02-01 00:00:00|389  |
|2024       |3           |Maret            |2024-03-01 00:00:00|453  |
|2024       |4           |April            |2024-04-01 00:00:00|492  |
|2024       |5           |Mei              |2024-05-01 00:00:00|620  |
|2024       |6           |Juni             |2024-06-01 00:00:00|697  |
|2024       |7           |Juli             |2024-07-01 00:00:00|1270 |
|2024       |8           |Agustus          |2024-08-01 00:00:00|1293 |
|2024       |9           |September        |2024-09-01 00:00:00|1123 |
|2024       |10          |Oktober          |2024-10-01 00:00:00|1217 |
|2024       |11          |November         |2024-11-01 00:00:00|1037 |
|2024 

## Distribusi Status Pesanan dengan Spark

In [44]:
if "Status Pesanan" in spark_df.columns:
    (
        spark_df
        .groupBy("Status Pesanan")
        .count()
        .orderBy(col("count").desc())
        .show(truncate=False)
    )
else:
    print("Kolom Status Pesanan tidak ditemukan.")

+--------------------------------------------------------------------------------------+-----+
|Status Pesanan                                                                        |count|
+--------------------------------------------------------------------------------------+-----+
|Selesai                                                                               |17416|
|Batal                                                                                 |2738 |
|Sedang Dikirim                                                                        |59   |
|Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-08.|38   |
|Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-07.|34   |
|Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-06.|34   |
|Telah Dikirim                                                                         |30   |
|Pesanan diterima, namun Pembeli masih dapat menga

## Top Kategori Produk dengan Spark

In [45]:
if "product_categories" in spark_df.columns:
    (
        spark_df
        .groupBy("product_categories")
        .count()
        .orderBy(col("count").desc())
        .show(20, truncate=False)
    )
else:
    print("Kolom product_categories tidak ditemukan.")

+----------------------+-----+
|product_categories    |count|
+----------------------+-----+
|Celengan              |5342 |
|Mangkok Sambal / Saus |3520 |
|Aksesoris Pintu       |2721 |
|Nampan / Tray         |1300 |
|Seal / Baut / Roof    |652  |
|Baskom / Mangkok Besar|573  |
|Rak / Rak Serbaguna   |568  |
|Lunch Box / Rantang   |522  |
|Keranjang             |493  |
|Other                 |493  |
|Tempat Nasi           |443  |
|Saringan / Strainer   |274  |
|Botol / Gelas / Mug   |240  |
|Teko / Jug            |199  |
|Peralatan Kamar Mandi |189  |
|Peralatan Makan       |180  |
|Pengolah Bumbu / Sayur|155  |
|Piring                |141  |
|Toples / Sealware     |127  |
|Stempel / Alat Kantor |118  |
+----------------------+-----+
only showing top 20 rows


## Top Provinsi dengan Spark

In [46]:
if "Provinsi" in spark_df.columns:
    (
        spark_df
        .groupBy("Provinsi")
        .count()
        .orderBy(col("count").desc())
        .show(20, truncate=False)
    )
else:
    print("Kolom Provinsi tidak ditemukan.")

+------------------------------+-----+
|Provinsi                      |count|
+------------------------------+-----+
|JAWA BARAT                    |6497 |
|BANTEN                        |3528 |
|DKI JAKARTA                   |2811 |
|JAWA TIMUR                    |1587 |
|JAWA TENGAH                   |1477 |
|SUMATERA SELATAN              |625  |
|LAMPUNG                       |483  |
|JAMBI                         |338  |
|RIAU                          |333  |
|SUMATERA UTARA                |306  |
|SUMATERA BARAT                |274  |
|SULAWESI SELATAN              |254  |
|DI YOGYAKARTA                 |251  |
|KALIMANTAN BARAT              |207  |
|BALI                          |192  |
|KALIMANTAN SELATAN            |151  |
|BENGKULU                      |139  |
|KALIMANTAN TENGAH             |131  |
|KALIMANTAN TIMUR              |130  |
|NANGGROE ACEH DARUSSALAM (NAD)|129  |
+------------------------------+-----+
only showing top 20 rows


## Top Metode Pembayaran dengan Spark

In [47]:
if "Metode Pembayaran" in spark_df.columns:
    (
        spark_df
        .groupBy("Metode Pembayaran")
        .count()
        .orderBy(col("count").desc())
        .show(20, truncate=False)
    )
else:
    print("Kolom Metode Pembayaran tidak ditemukan.")

+-------------------------+-----+
|Metode Pembayaran        |count|
+-------------------------+-----+
|COD (Bayar di Tempat)    |11332|
|Saldo ShopeePay          |3599 |
|Online Payment           |3109 |
|SPayLater                |1456 |
|SeaBank Bayar Instan     |584  |
|Kartu Kredit/Debit       |222  |
|Indomaret/i.Saku         |42   |
|Alfamart/Alfamidi/Dan+Dan|35   |
|Pembayaran dibebaskan    |18   |
|BCA OneKlik              |3    |
|Cicilan Kartu Kredit     |3    |
|Mitra Shopee             |1    |
+-------------------------+-----+



## Mengecek Ulang Dataset gabungan

In [50]:
print("Penyimpanan ulang menggunakan Spark dilewati.")
print("Dataset gabungan sudah berhasil disimpan menggunakan Pandas pada path:")
print(combined_csv_path)

Penyimpanan ulang menggunakan Spark dilewati.
Dataset gabungan sudah berhasil disimpan menggunakan Pandas pada path:
e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\datasets\combined\indonesia_ecommerce_sales_2024_2025_combined.csv


In [51]:
df_check = pd.read_csv(combined_csv_path)

print("Jumlah baris dataset awal       :", df_combined.shape[0])
print("Jumlah baris CSV hasil simpan   :", df_check.shape[0])
print("Jumlah kolom dataset awal       :", df_combined.shape[1])
print("Jumlah kolom CSV hasil simpan   :", df_check.shape[1])

if df_combined.shape == df_check.shape:
    print("Validasi berhasil: ukuran dataset sebelum dan sesudah disimpan sama.")
else:
    print("Validasi perlu dicek: ukuran dataset berbeda.")

Jumlah baris dataset awal       : 20404
Jumlah baris CSV hasil simpan   : 20404
Jumlah kolom dataset awal       : 27
Jumlah kolom CSV hasil simpan   : 27
Validasi berhasil: ukuran dataset sebelum dan sesudah disimpan sama.


In [52]:
spark_row_count = spark_df.count()
pandas_row_count = df_combined.shape[0]

print("Jumlah baris Pandas DataFrame :", pandas_row_count)
print("Jumlah baris Spark DataFrame  :", spark_row_count)

if pandas_row_count == spark_row_count:
    print("Validasi berhasil: dataset CSV dapat dibaca oleh PySpark dengan jumlah baris yang sama.")
else:
    print("Validasi perlu dicek: jumlah baris Pandas dan Spark berbeda.")

Jumlah baris Pandas DataFrame : 20404
Jumlah baris Spark DataFrame  : 20404
Validasi berhasil: dataset CSV dapat dibaca oleh PySpark dengan jumlah baris yang sama.


In [53]:
print("RINGKASAN NOTEBOOK 1")
print("=" * 70)
print(f"Jumlah file Excel yang dibaca     : {len(excel_files)}")
print(f"Jumlah baris dataset gabungan     : {df_combined.shape[0]}")
print(f"Jumlah kolom dataset gabungan     : {df_combined.shape[1]}")
print(f"Jumlah baris Spark DataFrame      : {spark_df.count()}")
print(f"File CSV gabungan                 : {combined_csv_path}")
print("=" * 70)

RINGKASAN NOTEBOOK 1
Jumlah file Excel yang dibaca     : 23
Jumlah baris dataset gabungan     : 20404
Jumlah kolom dataset gabungan     : 27
Jumlah baris Spark DataFrame      : 20404
File CSV gabungan                 : e:\DOCUMENTS\PERKULIAHAN\SEMESTER 8\JOKI\TUGAS\Big Data\Tubes\code\datasets\combined\indonesia_ecommerce_sales_2024_2025_combined.csv
